# 🤖 Gesture-Controlled Robot — Transformer Training Notebook
## Phase 2: Advanced AI & Sequence Modeling

**Project:** AI-Powered Gesture-Controlled Robot with Blockchain Security
**Author:** Shubham Shukla
**Date:** March 2026

This notebook trains a **Transformer Encoder** model on hand landmark sequences
to classify dynamic gestures for real-time robot control.

### Architecture Overview
- **Input:** Sequences of 30-60 frames, each frame = 63 features (21 landmarks × 3 coords)
- **Model:** Transformer Encoder with positional encoding
- **Output:** Gesture class probabilities
- **Target Latency:** < 300ms per prediction

### Supported Gestures
| Gesture | Robot Action |
|---------|-------------|
| Swipe Left | Turn Left |
| Swipe Right | Turn Right |
| Push Forward | Move Forward |
| Pull Back | Move Backward |
| Open Palm | Stop |
| Fist | Emergency Stop |
| Thumbs Up | Speed Up |
| Thumbs Down | Slow Down |

## 1. Environment Setup

First, install all required dependencies and configure the GPU runtime.

In [ ]:
# ============================================================
# Cell 1: Install Dependencies & Check GPU
# ============================================================

!pip install torch torchvision torchaudio --quiet
!pip install scikit-learn matplotlib seaborn tqdm --quiet
!pip install tensorboard --quiet

import torch
import numpy as np
import os
import json
import time
import math
import csv
from pathlib import Path
from datetime import datetime

# Check GPU availability
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'🖥️  PyTorch Version: {torch.__version__}')
print(f'🎮 Device: {device}')
if torch.cuda.is_available():
    print(f'🚀 GPU: {torch.cuda.get_device_name(0)}')
    print(f'💾 GPU Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print('⚠️  No GPU detected. Go to Runtime > Change runtime type > GPU')

## 2. Configuration & Hyperparameters

Define all training parameters in one place for easy experimentation.

In [ ]:
# ============================================================
# Cell 2: Configuration
# ============================================================

class Config:
    """Training configuration — all hyperparameters in one place."""
    
    # Dataset
    GESTURE_CLASSES = [
        'swipe_left', 'swipe_right', 'push_forward',
        'pull_back', 'open_palm', 'fist',
        'thumbs_up', 'thumbs_down'
    ]
    NUM_CLASSES = len(GESTURE_CLASSES)
    SEQUENCE_LENGTH = 45          # frames per gesture (30-60 range)
    NUM_LANDMARKS = 21            # MediaPipe hand landmarks
    COORDS_PER_LANDMARK = 3       # X, Y, Z
    INPUT_DIM = NUM_LANDMARKS * COORDS_PER_LANDMARK  # 63
    
    # Transformer Architecture
    D_MODEL = 128                 # Embedding dimension
    N_HEADS = 8                   # Attention heads
    N_ENCODER_LAYERS = 4          # Transformer encoder layers
    DIM_FEEDFORWARD = 512         # FFN hidden dimension
    DROPOUT = 0.15                # Dropout rate
    
    # Training
    BATCH_SIZE = 32
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-5
    NUM_EPOCHS = 100
    EARLY_STOPPING_PATIENCE = 15
    LR_SCHEDULER_PATIENCE = 7
    LR_SCHEDULER_FACTOR = 0.5
    WARMUP_EPOCHS = 5
    
    # Data Split
    TRAIN_RATIO = 0.7
    VAL_RATIO = 0.15
    TEST_RATIO = 0.15
    
    # Augmentation
    AUG_NOISE_STD = 0.01
    AUG_SCALE_RANGE = (0.9, 1.1)
    AUG_TIME_WARP_FACTOR = 0.1
    AUG_ROTATION_RANGE = 15  # degrees
    
    # Paths
    DATA_DIR = './data'
    MODEL_DIR = './models'
    LOG_DIR = './logs'
    
    # Performance Targets (PG-level metrics)
    TARGET_LATENCY_MS = 300       # Max inference time
    TARGET_ACCURACY = 0.92        # Min accuracy
    STABILITY_WINDOW = 5          # Frames for prediction stability

config = Config()

# Create directories
os.makedirs(config.DATA_DIR, exist_ok=True)
os.makedirs(config.MODEL_DIR, exist_ok=True)
os.makedirs(config.LOG_DIR, exist_ok=True)

print('📋 Configuration:')
print(f'   Classes: {config.NUM_CLASSES}')
print(f'   Sequence Length: {config.SEQUENCE_LENGTH} frames')
print(f'   Input Dim: {config.INPUT_DIM}')
print(f'   Model Dim: {config.D_MODEL}')
print(f'   Attention Heads: {config.N_HEADS}')
print(f'   Encoder Layers: {config.N_ENCODER_LAYERS}')
print(f'   Batch Size: {config.BATCH_SIZE}')
print(f'   Learning Rate: {config.LEARNING_RATE}')

## 3. Data Loading & Preprocessing

Load the landmark CSV/JSON data collected by `data_collector.py` and prepare
PyTorch datasets with proper normalization and augmentation.

In [ ]:
# ============================================================
# Cell 3: Data Loading Utilities
# ============================================================

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import glob


def load_landmark_csv(file_path):
    """Load landmark data from CSV file.
    
    Expected CSV format:
    frame_idx, lm0_x, lm0_y, lm0_z, lm1_x, lm1_y, lm1_z, ..., lm20_x, lm20_y, lm20_z
    """
    frames = []
    with open(file_path, 'r') as f:
        reader = csv.reader(f)
        header = next(reader, None)  # Skip header
        for row in reader:
            # Skip frame_idx column, take landmark coords
            coords = [float(x) for x in row[1:]]
            frames.append(coords)
    return np.array(frames, dtype=np.float32)


def load_landmark_json(file_path):
    """Load landmark data from JSON file.
    
    Expected JSON format:
    {
        "gesture": "swipe_left",
        "frames": [[x0,y0,z0, x1,y1,z1, ...], ...]
    }
    """
    with open(file_path, 'r') as f:
        data = json.load(f)
    return np.array(data['frames'], dtype=np.float32)


def normalize_landmarks(sequence):
    """Normalize landmarks to be invariant to hand distance and position.
    
    Strategy:
    1. Center on wrist (landmark 0)
    2. Scale by max distance from wrist
    3. This makes the model invariant to hand size and distance from camera
    """
    normalized = sequence.copy()
    num_frames = normalized.shape[0]
    
    for i in range(num_frames):
        frame = normalized[i].reshape(21, 3)
        
        # Center on wrist (landmark 0)
        wrist = frame[0].copy()
        frame -= wrist
        
        # Scale by max distance from wrist
        distances = np.linalg.norm(frame, axis=1)
        max_dist = np.max(distances)
        if max_dist > 1e-6:
            frame /= max_dist
        
        normalized[i] = frame.flatten()
    
    return normalized


def pad_or_truncate(sequence, target_length):
    """Ensure all sequences have the same length."""
    current_length = len(sequence)
    
    if current_length == target_length:
        return sequence
    elif current_length > target_length:
        # Uniform sampling to preserve temporal structure
        indices = np.linspace(0, current_length - 1, target_length, dtype=int)
        return sequence[indices]
    else:
        # Pad with last frame (more natural than zero-padding)
        padding = np.tile(sequence[-1:], (target_length - current_length, 1))
        return np.vstack([sequence, padding])


print('✅ Data loading utilities defined')

In [ ]:
# ============================================================
# Cell 4: Data Augmentation
# ============================================================


class GestureAugmentor:
    """Data augmentation for hand landmark sequences.
    
    Augmentations applied:
    1. Gaussian noise injection
    2. Random scaling
    3. Time warping (speed variation)
    4. Random rotation around Z-axis
    5. Temporal jittering
    """
    
    def __init__(self, config):
        self.noise_std = config.AUG_NOISE_STD
        self.scale_range = config.AUG_SCALE_RANGE
        self.time_warp_factor = config.AUG_TIME_WARP_FACTOR
        self.rotation_range = config.AUG_ROTATION_RANGE
    
    def add_noise(self, sequence):
        """Add Gaussian noise to simulate sensor imprecision."""
        noise = np.random.normal(0, self.noise_std, sequence.shape)
        return sequence + noise.astype(np.float32)
    
    def random_scale(self, sequence):
        """Random uniform scaling to simulate distance variation."""
        scale = np.random.uniform(*self.scale_range)
        return sequence * scale
    
    def time_warp(self, sequence):
        """Warp time axis to simulate speed variations."""
        seq_len = len(sequence)
        # Create warped time indices
        warp = np.random.uniform(
            1 - self.time_warp_factor,
            1 + self.time_warp_factor,
            seq_len
        )
        warp_cumsum = np.cumsum(warp)
        warp_cumsum = warp_cumsum / warp_cumsum[-1] * (seq_len - 1)
        
        # Interpolate
        warped = np.zeros_like(sequence)
        for feat_idx in range(sequence.shape[1]):
            warped[:, feat_idx] = np.interp(
                warp_cumsum,
                np.arange(seq_len),
                sequence[:, feat_idx]
            )
        return warped.astype(np.float32)
    
    def rotate_z(self, sequence):
        """Random rotation around Z-axis (camera axis)."""
        angle = np.radians(np.random.uniform(-self.rotation_range, self.rotation_range))
        cos_a, sin_a = np.cos(angle), np.sin(angle)
        
        rotated = sequence.copy()
        for i in range(len(rotated)):
            frame = rotated[i].reshape(21, 3)
            x_new = frame[:, 0] * cos_a - frame[:, 1] * sin_a
            y_new = frame[:, 0] * sin_a + frame[:, 1] * cos_a
            frame[:, 0] = x_new
            frame[:, 1] = y_new
            rotated[i] = frame.flatten()
        return rotated
    
    def augment(self, sequence):
        """Apply random combination of augmentations."""
        augmented = sequence.copy()
        
        if np.random.random() > 0.5:
            augmented = self.add_noise(augmented)
        if np.random.random() > 0.5:
            augmented = self.random_scale(augmented)
        if np.random.random() > 0.5:
            augmented = self.time_warp(augmented)
        if np.random.random() > 0.5:
            augmented = self.rotate_z(augmented)
        
        return augmented


augmentor = GestureAugmentor(config)
print('✅ Data augmentation pipeline ready')

In [ ]:
# ============================================================
# Cell 5: PyTorch Dataset
# ============================================================


class GestureDataset(Dataset):
    """PyTorch Dataset for gesture landmark sequences."""
    
    def __init__(self, sequences, labels, config, augment=False):
        self.sequences = sequences
        self.labels = labels
        self.config = config
        self.augment = augment
        self.augmentor = GestureAugmentor(config) if augment else None
    
    def __len__(self):
        return len(self.sequences)
    
    def __getitem__(self, idx):
        sequence = self.sequences[idx].copy()
        label = self.labels[idx]
        
        # Apply augmentation during training
        if self.augment and self.augmentor:
            sequence = self.augmentor.augment(sequence)
        
        return (
            torch.FloatTensor(sequence),
            torch.LongTensor([label]).squeeze()
        )


print('✅ GestureDataset class defined')

In [ ]:
# ============================================================
# Cell 6: Load & Prepare Data
# ============================================================

def load_dataset(data_dir, config):
    """Load all gesture data from directory structure.
    
    Expected structure:
    data/
      swipe_left/
        sample_001.csv
        sample_002.csv
      swipe_right/
        ...
    """
    all_sequences = []
    all_labels = []
    label_encoder = LabelEncoder()
    label_encoder.fit(config.GESTURE_CLASSES)
    
    for gesture_name in config.GESTURE_CLASSES:
        gesture_dir = os.path.join(data_dir, gesture_name)
        if not os.path.exists(gesture_dir):
            print(f'  ⚠️  Directory not found: {gesture_dir}')
            continue
        
        # Load CSV files
        csv_files = glob.glob(os.path.join(gesture_dir, '*.csv'))
        json_files = glob.glob(os.path.join(gesture_dir, '*.json'))
        
        for f in csv_files:
            try:
                seq = load_landmark_csv(f)
                seq = normalize_landmarks(seq)
                seq = pad_or_truncate(seq, config.SEQUENCE_LENGTH)
                all_sequences.append(seq)
                all_labels.append(label_encoder.transform([gesture_name])[0])
            except Exception as e:
                print(f'  ❌ Error loading {f}: {e}')
        
        for f in json_files:
            try:
                seq = load_landmark_json(f)
                seq = normalize_landmarks(seq)
                seq = pad_or_truncate(seq, config.SEQUENCE_LENGTH)
                all_sequences.append(seq)
                all_labels.append(label_encoder.transform([gesture_name])[0])
            except Exception as e:
                print(f'  ❌ Error loading {f}: {e}')
        
        count = len(csv_files) + len(json_files)
        print(f'  ✅ {gesture_name}: {count} samples loaded')
    
    return np.array(all_sequences), np.array(all_labels), label_encoder


def generate_synthetic_data(config, samples_per_class=100):
    """Generate synthetic training data for testing the pipeline.
    
    Each gesture class gets a unique motion pattern:
    - swipe_left: X decreases over time
    - swipe_right: X increases over time
    - push_forward: Z decreases (hand moves toward camera)
    - pull_back: Z increases
    - open_palm: fingers spread (high variance)
    - fist: fingers close (low variance)
    - thumbs_up: Y of thumb increases
    - thumbs_down: Y of thumb decreases
    """
    print('\n🔧 Generating synthetic training data...')
    
    all_sequences = []
    all_labels = []
    seq_len = config.SEQUENCE_LENGTH
    n_features = config.INPUT_DIM
    
    for class_idx, gesture in enumerate(config.GESTURE_CLASSES):
        for sample in range(samples_per_class):
            # Base hand pose with slight randomness
            base = np.random.randn(1, n_features).astype(np.float32) * 0.1
            sequence = np.tile(base, (seq_len, 1))
            
            # Time vector
            t = np.linspace(0, 1, seq_len).reshape(-1, 1)
            
            # Add class-specific temporal patterns
            if gesture == 'swipe_left':
                # X coords (indices 0, 3, 6, ...) decrease
                for lm in range(21):
                    sequence[:, lm*3] -= t.flatten() * 0.5
            elif gesture == 'swipe_right':
                for lm in range(21):
                    sequence[:, lm*3] += t.flatten() * 0.5
            elif gesture == 'push_forward':
                for lm in range(21):
                    sequence[:, lm*3+2] -= t.flatten() * 0.4
            elif gesture == 'pull_back':
                for lm in range(21):
                    sequence[:, lm*3+2] += t.flatten() * 0.4
            elif gesture == 'open_palm':
                spread = t * 0.3
                for lm in range(5, 21):
                    sequence[:, lm*3] += spread.flatten() * (lm % 3 - 1)
                    sequence[:, lm*3+1] += spread.flatten() * 0.5
            elif gesture == 'fist':
                close = t * 0.3
                for lm in range(5, 21):
                    sequence[:, lm*3] *= (1 - close.flatten())
                    sequence[:, lm*3+1] *= (1 - close.flatten())
            elif gesture == 'thumbs_up':
                # Thumb tip (landmark 4) Y goes up
                sequence[:, 4*3+1] += t.flatten() * 0.6
            elif gesture == 'thumbs_down':
                sequence[:, 4*3+1] -= t.flatten() * 0.6
            
            # Add noise
            sequence += np.random.randn(*sequence.shape).astype(np.float32) * 0.02
            
            all_sequences.append(sequence)
            all_labels.append(class_idx)
        
        print(f'  ✅ {gesture}: {samples_per_class} synthetic samples')
    
    return np.array(all_sequences), np.array(all_labels)


# Try loading real data first, fall back to synthetic
if os.path.exists(config.DATA_DIR) and any(
    os.path.isdir(os.path.join(config.DATA_DIR, g))
    for g in config.GESTURE_CLASSES
):
    print('📂 Loading real dataset...')
    sequences, labels, label_enc = load_dataset(config.DATA_DIR, config)
else:
    print('📂 No real data found. Generating synthetic dataset...')
    sequences, labels = generate_synthetic_data(config, samples_per_class=150)
    label_enc = LabelEncoder()
    label_enc.fit(config.GESTURE_CLASSES)

print(f'\n📊 Dataset Summary:')
print(f'   Total samples: {len(sequences)}')
print(f'   Sequence shape: {sequences[0].shape}')
print(f'   Classes: {config.NUM_CLASSES}')

# Split data
X_train, X_temp, y_train, y_temp = train_test_split(
    sequences, labels,
    test_size=(config.VAL_RATIO + config.TEST_RATIO),
    random_state=42,
    stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=config.TEST_RATIO / (config.VAL_RATIO + config.TEST_RATIO),
    random_state=42,
    stratify=y_temp
)

print(f'   Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}')

# Create DataLoaders
train_dataset = GestureDataset(X_train, y_train, config, augment=True)
val_dataset = GestureDataset(X_val, y_val, config, augment=False)
test_dataset = GestureDataset(X_test, y_test, config, augment=False)

train_loader = DataLoader(train_dataset, batch_size=config.BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=config.BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=config.BATCH_SIZE, shuffle=False)

print('\n✅ DataLoaders ready')

## 4. Transformer Encoder Model

The core of our gesture recognition system — a Transformer Encoder that processes
temporal sequences of hand landmarks to classify dynamic gestures.

### Why Transformer over LSTM?
- **Parallel processing** of all frames (faster training)
- **Self-attention** captures long-range temporal dependencies
- **Positional encoding** preserves temporal order
- **State-of-the-art** for sequence analysis tasks

In [ ]:
# ============================================================
# Cell 7: Positional Encoding
# ============================================================

import torch.nn as nn


class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding for temporal awareness.
    
    Injects information about the position of each frame in the sequence,
    allowing the Transformer to understand temporal order.
    """
    
    def __init__(self, d_model, max_len=500, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        
        # Create positional encoding matrix
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        
        self.register_buffer('pe', pe)
    
    def forward(self, x):
        """x shape: (batch, seq_len, d_model)"""
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)


print('✅ PositionalEncoding defined')

In [ ]:
# ============================================================
# Cell 8: Gesture Transformer Model
# ============================================================


class GestureTransformer(nn.Module):
    """Transformer Encoder for gesture sequence classification.
    
    Architecture:
    1. Linear projection: 63 → d_model
    2. Positional encoding
    3. N × Transformer Encoder layers
    4. Global average pooling
    5. Classification head
    """
    
    def __init__(self, config):
        super().__init__()
        
        self.config = config
        
        # Input projection
        self.input_projection = nn.Sequential(
            nn.Linear(config.INPUT_DIM, config.D_MODEL),
            nn.LayerNorm(config.D_MODEL),
            nn.ReLU(),
            nn.Dropout(config.DROPOUT)
        )
        
        # Positional encoding
        self.pos_encoder = PositionalEncoding(
            config.D_MODEL,
            max_len=config.SEQUENCE_LENGTH,
            dropout=config.DROPOUT
        )
        
        # Transformer Encoder
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=config.D_MODEL,
            nhead=config.N_HEADS,
            dim_feedforward=config.DIM_FEEDFORWARD,
            dropout=config.DROPOUT,
            activation='gelu',
            batch_first=True,
            norm_first=True  # Pre-norm for better training stability
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=config.N_ENCODER_LAYERS,
            norm=nn.LayerNorm(config.D_MODEL)
        )
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(config.D_MODEL, config.D_MODEL // 2),
            nn.GELU(),
            nn.Dropout(config.DROPOUT),
            nn.Linear(config.D_MODEL // 2, config.NUM_CLASSES)
        )
        
        # Initialize weights
        self._init_weights()
    
    def _init_weights(self):
        """Xavier uniform initialization for better convergence."""
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)
    
    def forward(self, x):
        """
        Args:
            x: (batch, seq_len, 63) landmark sequences
        Returns:
            logits: (batch, num_classes)
        """
        # Project input to model dimension
        x = self.input_projection(x)  # (B, T, D)
        
        # Add positional encoding
        x = self.pos_encoder(x)  # (B, T, D)
        
        # Transformer encoding
        x = self.transformer_encoder(x)  # (B, T, D)
        
        # Global average pooling over time dimension
        x = x.mean(dim=1)  # (B, D)
        
        # Classify
        logits = self.classifier(x)  # (B, C)
        
        return logits
    
    def count_parameters(self):
        """Count trainable parameters."""
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


# Instantiate model
model = GestureTransformer(config).to(device)

print(f'\n🧠 GestureTransformer Architecture:')
print(f'   Trainable Parameters: {model.count_parameters():,}')
print(f'   Device: {device}')
print(f'\n{model}')

## 5. Training Loop

Complete training pipeline with:
- Warmup + ReduceLROnPlateau scheduler
- Early stopping
- Gradient clipping
- TensorBoard logging
- Best model checkpointing

In [ ]:
# ============================================================
# Cell 9: Training Utilities
# ============================================================

from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt
import seaborn as sns


class EarlyStopping:
    """Early stopping to prevent overfitting."""
    
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.should_stop = False
    
    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0
        return self.should_stop


class WarmupScheduler:
    """Linear warmup followed by ReduceLROnPlateau."""
    
    def __init__(self, optimizer, warmup_epochs, base_lr):
        self.optimizer = optimizer
        self.warmup_epochs = warmup_epochs
        self.base_lr = base_lr
        self.current_epoch = 0
    
    def step(self, epoch):
        self.current_epoch = epoch
        if epoch < self.warmup_epochs:
            lr = self.base_lr * (epoch + 1) / self.warmup_epochs
            for param_group in self.optimizer.param_groups:
                param_group['lr'] = lr
            return lr
        return None  # Let ReduceLROnPlateau handle it


print('✅ Training utilities defined')

In [ ]:
# ============================================================
# Cell 10: Training Loop
# ============================================================

def train_model(model, train_loader, val_loader, config, device):
    """Full training pipeline with logging and checkpointing."""
    
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=config.LEARNING_RATE,
        weight_decay=config.WEIGHT_DECAY
    )
    
    # Schedulers
    warmup = WarmupScheduler(optimizer, config.WARMUP_EPOCHS, config.LEARNING_RATE)
    plateau_scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode='min',
        factor=config.LR_SCHEDULER_FACTOR,
        patience=config.LR_SCHEDULER_PATIENCE,
        verbose=True
    )
    early_stopping = EarlyStopping(patience=config.EARLY_STOPPING_PATIENCE)
    
    # TensorBoard
    writer = SummaryWriter(config.LOG_DIR)
    
    # Training history
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'lr': []
    }
    best_val_loss = float('inf')
    best_val_acc = 0.0
    
    print(f'\n🚀 Starting training for {config.NUM_EPOCHS} epochs...')
    print(f'   Warmup: {config.WARMUP_EPOCHS} epochs')
    print(f'   Early stopping patience: {config.EARLY_STOPPING_PATIENCE}')
    print('=' * 70)
    
    for epoch in range(config.NUM_EPOCHS):
        # === TRAINING PHASE ===
        model.train()
        train_loss = 0.0
        train_preds = []
        train_targets = []
        
        # Warmup LR
        warmup.step(epoch)
        current_lr = optimizer.param_groups[0]['lr']
        
        pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{config.NUM_EPOCHS}', leave=False)
        for batch_x, batch_y in pbar:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)
            
            # Forward pass
            optimizer.zero_grad()
            logits = model(batch_x)
            loss = criterion(logits, batch_y)
            
            # Backward pass with gradient clipping
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            
            train_loss += loss.item()
            preds = logits.argmax(dim=1).cpu().numpy()
            train_preds.extend(preds)
            train_targets.extend(batch_y.cpu().numpy())
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        
        avg_train_loss = train_loss / len(train_loader)
        train_acc = accuracy_score(train_targets, train_preds)
        
        # === VALIDATION PHASE ===
        model.eval()
        val_loss = 0.0
        val_preds = []
        val_targets = []
        
        with torch.no_grad():
            for batch_x, batch_y in val_loader:
                batch_x = batch_x.to(device)
                batch_y = batch_y.to(device)
                
                logits = model(batch_x)
                loss = criterion(logits, batch_y)
                
                val_loss += loss.item()
                preds = logits.argmax(dim=1).cpu().numpy()
                val_preds.extend(preds)
                val_targets.extend(batch_y.cpu().numpy())
        
        avg_val_loss = val_loss / len(val_loader)
        val_acc = accuracy_score(val_targets, val_preds)
        
        # Update schedulers
        if epoch >= config.WARMUP_EPOCHS:
            plateau_scheduler.step(avg_val_loss)
        
        # Log to history
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)
        
        # TensorBoard logging
        writer.add_scalars('Loss', {'train': avg_train_loss, 'val': avg_val_loss}, epoch)
        writer.add_scalars('Accuracy', {'train': train_acc, 'val': val_acc}, epoch)
        writer.add_scalar('LR', current_lr, epoch)
        
        # Print progress
        improved = ''
        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            best_val_acc = val_acc
            improved = ' ⭐ BEST'
            # Save best model
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': best_val_loss,
                'val_acc': best_val_acc,
                'config': vars(config),
            }, os.path.join(config.MODEL_DIR, 'best_gesture_transformer.pth'))
        
        print(
            f'Epoch {epoch+1:3d}/{config.NUM_EPOCHS} | '
            f'Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f} | '
            f'Val Loss: {avg_val_loss:.4f} | Val Acc: {val_acc:.4f} | '
            f'LR: {current_lr:.2e}{improved}'
        )
        
        # Early stopping check
        if early_stopping(avg_val_loss):
            print(f'\n⏹️  Early stopping triggered at epoch {epoch+1}')
            break
    
    writer.close()
    print(f'\n✅ Training complete!')
    print(f'   Best Val Loss: {best_val_loss:.4f}')
    print(f'   Best Val Accuracy: {best_val_acc:.4f}')
    
    return history


# Train the model
history = train_model(model, train_loader, val_loader, config, device)

## 6. Training Visualization

Plot training curves to analyze model convergence and detect overfitting.

In [ ]:
# ============================================================
# Cell 11: Plot Training History
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Loss curves
axes[0].plot(history['train_loss'], label='Train Loss', color='#2196F3', linewidth=2)
axes[0].plot(history['val_loss'], label='Val Loss', color='#F44336', linewidth=2)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('📉 Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Accuracy curves
axes[1].plot(history['train_acc'], label='Train Acc', color='#4CAF50', linewidth=2)
axes[1].plot(history['val_acc'], label='Val Acc', color='#FF9800', linewidth=2)
axes[1].axhline(y=config.TARGET_ACCURACY, color='red', linestyle='--', label=f'Target ({config.TARGET_ACCURACY})')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('📈 Training & Validation Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Learning rate
axes[2].plot(history['lr'], color='#9C27B0', linewidth=2)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('🔧 Learning Rate Schedule')
axes[2].set_yscale('log')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(config.MODEL_DIR, 'training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()
print('📊 Training curves saved')

## 7. Model Evaluation

Comprehensive evaluation on the held-out test set with:
- Classification report (precision, recall, F1 per class)
- Confusion matrix
- Inference latency measurement
- Prediction stability analysis

In [ ]:
# ============================================================
# Cell 12: Test Set Evaluation
# ============================================================

# Load best model
checkpoint = torch.load(
    os.path.join(config.MODEL_DIR, 'best_gesture_transformer.pth'),
    map_location=device
)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()

print(f'📦 Loaded best model from epoch {checkpoint["epoch"] + 1}')
print(f'   Val Loss: {checkpoint["val_loss"]:.4f}')
print(f'   Val Acc: {checkpoint["val_acc"]:.4f}')

# Evaluate on test set
all_preds = []
all_targets = []
all_probs = []

with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x = batch_x.to(device)
        logits = model(batch_x)
        probs = torch.softmax(logits, dim=1)
        preds = logits.argmax(dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(batch_y.numpy())
        all_probs.extend(probs.cpu().numpy())

# Classification Report
print('\n' + '=' * 70)
print('📋 CLASSIFICATION REPORT')
print('=' * 70)
print(classification_report(
    all_targets, all_preds,
    target_names=config.GESTURE_CLASSES,
    digits=4
))

# Overall metrics
test_acc = accuracy_score(all_targets, all_preds)
test_f1 = f1_score(all_targets, all_preds, average='weighted')
print(f'\n🎯 Test Accuracy: {test_acc:.4f}')
print(f'🎯 Test F1 Score: {test_f1:.4f}')
print(f'🎯 Target Accuracy: {config.TARGET_ACCURACY} → {"✅ PASSED" if test_acc >= config.TARGET_ACCURACY else "❌ BELOW TARGET"}')

# Confusion Matrix
cm = confusion_matrix(all_targets, all_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=config.GESTURE_CLASSES,
    yticklabels=config.GESTURE_CLASSES
)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('🔍 Confusion Matrix — Gesture Classification')
plt.tight_layout()
plt.savefig(os.path.join(config.MODEL_DIR, 'confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# Cell 13: Inference Latency & Prediction Stability
# ============================================================

print('⏱️  Measuring Inference Latency...')
print('=' * 50)

# Warm up
dummy = torch.randn(1, config.SEQUENCE_LENGTH, config.INPUT_DIM).to(device)
for _ in range(10):
    _ = model(dummy)

# Measure latency
latencies = []
num_runs = 100

with torch.no_grad():
    for _ in range(num_runs):
        sample = torch.randn(1, config.SEQUENCE_LENGTH, config.INPUT_DIM).to(device)
        
        if device.type == 'cuda':
            torch.cuda.synchronize()
        start = time.perf_counter()
        
        _ = model(sample)
        
        if device.type == 'cuda':
            torch.cuda.synchronize()
        end = time.perf_counter()
        
        latencies.append((end - start) * 1000)  # ms

avg_latency = np.mean(latencies)
p95_latency = np.percentile(latencies, 95)
p99_latency = np.percentile(latencies, 99)

print(f'   Average Latency: {avg_latency:.2f} ms')
print(f'   P95 Latency:     {p95_latency:.2f} ms')
print(f'   P99 Latency:     {p99_latency:.2f} ms')
print(f'   Target:          < {config.TARGET_LATENCY_MS} ms')
print(f'   Status:          {"✅ PASSED" if p95_latency < config.TARGET_LATENCY_MS else "❌ TOO SLOW"}')

# Prediction Stability Test
print(f'\n🔄 Prediction Stability Test (window={config.STABILITY_WINDOW})...')
print('=' * 50)

stability_scores = []

with torch.no_grad():
    for i in range(min(50, len(X_test))):
        # Simulate consecutive predictions with slight noise
        base_seq = torch.FloatTensor(X_test[i]).unsqueeze(0).to(device)
        predictions = []
        
        for _ in range(config.STABILITY_WINDOW):
            noisy = base_seq + torch.randn_like(base_seq) * 0.005
            pred = model(noisy).argmax(dim=1).item()
            predictions.append(pred)
        
        # Stability = fraction of consistent predictions
        most_common = max(set(predictions), key=predictions.count)
        stability = predictions.count(most_common) / len(predictions)
        stability_scores.append(stability)

avg_stability = np.mean(stability_scores)
print(f'   Average Stability: {avg_stability:.4f}')
print(f'   Min Stability:     {np.min(stability_scores):.4f}')
print(f'   Status:            {"✅ STABLE" if avg_stability > 0.9 else "⚠️  UNSTABLE - consider smoothing"}')

# Summary
print(f'\n{"=" * 70}')
print(f'📊 PG-LEVEL PERFORMANCE SUMMARY')
print(f'{"=" * 70}')
print(f'  Accuracy:    {test_acc:.4f}  (target: {config.TARGET_ACCURACY})')
print(f'  F1 Score:    {test_f1:.4f}')
print(f'  Latency:     {avg_latency:.2f} ms  (target: < {config.TARGET_LATENCY_MS} ms)')
print(f'  Stability:   {avg_stability:.4f}')
print(f'{"=" * 70}')

## 8. Export Model for Deployment

Export the trained model in formats suitable for:
- Local Python inference (PyTorch .pth)
- ONNX format (for cross-platform deployment)
- TorchScript (for C++ integration with Webots)

In [ ]:
# ============================================================
# Cell 14: Export Model
# ============================================================

# 1. Save full model state
torch.save({
    'model_state_dict': model.state_dict(),
    'config': vars(config),
    'gesture_classes': config.GESTURE_CLASSES,
    'test_accuracy': test_acc,
    'test_f1': test_f1,
    'avg_latency_ms': avg_latency,
    'stability': avg_stability,
    'training_date': datetime.now().isoformat(),
}, os.path.join(config.MODEL_DIR, 'gesture_transformer_final.pth'))
print('✅ Saved: gesture_transformer_final.pth')

# 2. Export to TorchScript
model.eval()
example_input = torch.randn(1, config.SEQUENCE_LENGTH, config.INPUT_DIM).to(device)
traced_model = torch.jit.trace(model, example_input)
traced_model.save(os.path.join(config.MODEL_DIR, 'gesture_transformer_traced.pt'))
print('✅ Saved: gesture_transformer_traced.pt (TorchScript)')

# 3. Export to ONNX
try:
    torch.onnx.export(
        model,
        example_input,
        os.path.join(config.MODEL_DIR, 'gesture_transformer.onnx'),
        export_params=True,
        opset_version=14,
        do_constant_folding=True,
        input_names=['landmark_sequence'],
        output_names=['gesture_logits'],
        dynamic_axes={
            'landmark_sequence': {0: 'batch_size'},
            'gesture_logits': {0: 'batch_size'}
        }
    )
    print('✅ Saved: gesture_transformer.onnx')
except Exception as e:
    print(f'⚠️  ONNX export failed: {e}')

# 4. Save label mapping
label_mapping = {i: name for i, name in enumerate(config.GESTURE_CLASSES)}
with open(os.path.join(config.MODEL_DIR, 'label_mapping.json'), 'w') as f:
    json.dump(label_mapping, f, indent=2)
print('✅ Saved: label_mapping.json')

print(f'\n📁 All models saved to {config.MODEL_DIR}/')
print('\n🎉 Training pipeline complete! Copy the models/ folder to your project.')

## 9. Download Models (Colab)

If running on Google Colab, download the trained models to your local machine.

In [ ]:
# ============================================================
# Cell 15: Download from Colab
# ============================================================

try:
    from google.colab import files
    
    # Zip models directory
    import shutil
    shutil.make_archive('trained_models', 'zip', '.', config.MODEL_DIR)
    
    # Download
    files.download('trained_models.zip')
    print('📥 Download started! Extract to your project\'s models/ directory.')
except ImportError:
    print('ℹ️  Not running on Colab. Models are saved locally in:', config.MODEL_DIR)
    print('   Copy the models/ folder to your gesture_robot_project/ directory.')